In [1]:
# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

import sys
import torch
from tqdm import tqdm
from nnsight import LanguageModel

sys.path.append("./mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder
from transformer_lens import utils

torch.set_grad_enabled(False)

/share/u/caden/.conda/envs/interp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir="./jbloom_dictionaries")

    save_path = f"./jbloom_dictionaries/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

Loaded dictionaries


In [ ]:
gpt = LanguageModel("openai-community/gpt2", device_map="cuda:0", dispatch=True)

print("Loaded GPT")

kwargs = {
    "load_in_4bit": True,
    "device_map": "cuda:0",
    "dispatch": True
}

mixtral = LanguageModel("mistralai/Mixtral-8x7B-Instruct-v0.1", **kwargs)

print("Loaded Mixtral in 4bit")

In [ ]:
messages = [
    {
        "role": "user", 
        "content": "For the rest of the conversation, return your answer in brackets, e.g. {ANSWER}. Just return an answer, nothing else. What is the capital of France?"
    },
    {
        "role": "assistant", 
        "content": "{Paris}"
    },
    {
        "role": "user", 
        "content": "What is a similar sentence to the following: 'I like pizza and big-ass big-scale ham.'? Return one sentence in brackets, nothing else."
    },
]

tok = mixtral.tokenizer.apply_chat_template(messages, return_tensors="pt")

In [ ]:
prompt = "the company spent $ 474 million on buy backs previous year - year , $ 40 million extension with the Marlins in million domestically and $ 128 million worldwide . The romantic comedy exceeded more than $ 30 million . Ċ Ċ Aud itors SUV production ; $ 150 million in its Romeo Engine Plant in Chicago of $ 100 million in used money that is of over $ 50 million. Ċ Ċ Related : 10 million to $ 100 million per year , necess itating to make over $ 395 million worldwide . Did you enjoy have grown to $ 32 million after spikes of ? 21 5 million to $ 39 million , worse than its estimate the Dodgers for $ 430 million . Ċ Ċ Advertisement Token Advertisement Feature activation +0.000 Ċ due to receive $ 3 million in a trust fund triggered of more than $ 1 million , but the House never"

In [ ]:
with gpt.trace(prompt):
    activation = gpt.transformer.h[0].attn.c_proj.input[0][0]

    loss, x_reconstruct, acts, l2_loss, l1_loss = attn_sae_list[0](activation)
    acts.save()

In [ ]:
from circuitsvis.tokens import colored_tokens

tokenized = gpt.tokenizer.encode(prompt)
str_tokens = [gpt.tokenizer.decode(i) for i in tokenized]
colored_tokens(str_tokens, acts[:,:,16][0].tolist())